# SIP-HyperGCN v4: Causal Concept Program
Thiết kế: `docs/architecture_v4.html`. Runtime: GPU. Chạy lần lượt từ trên xuống; mỗi giai đoạn lưu checkpoint lên Drive nên có thể chạy tiếp nếu Colab bị ngắt.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content
!if [ -d thesis ]; then git -C thesis pull; else git clone https://github.com/NguyenBaoHuy05/thesis.git; fi

/content


Already up to date.


## Cấu hình

In [ ]:
import os
%cd /content/thesis/SIP-HyperGCN

os.environ["PYTHONUNBUFFERED"] = "1"   # %%bash ghi vao pipe -> khong co dong nay thi print bi giu lai, khong thay log
os.environ["H"]  = "/content/thesis/GCCR-Skateformer-TreeRule/hypergcn"
os.environ["CK"] = "/content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt"
os.environ["D"]  = "/content/data/NTU60_CS.npz"
os.environ["V4"] = "/content/drive/MyDrive/BaoHuy/work_dir/ccp"      # ket qua v4 tren Drive
os.environ["COMMON"] = "--hypergcn_root {H} --checkpoint {CK} --data_path {D}".format(**os.environ)

DRIVE_DATA = "/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz"

In [7]:
import shutil
os.makedirs(os.path.dirname(os.environ["D"]), exist_ok=True)
if not os.path.exists(os.environ["D"]):
    shutil.copy(DRIVE_DATA, os.environ["D"])
for k in ["H", "CK", "D"]:
    print(k, os.environ[k], "OK" if os.path.exists(os.environ[k]) else "MISSING")

H /content/thesis/GCCR-Skateformer-TreeRule/hypergcn OK
CK /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt OK
D /content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz OK


## 0. Smoke test (vài batch, cả 3 giai đoạn + đánh giá)
Chạy cell này trước để bắt lỗi sớm (khoảng 3-5 phút). Kết quả nằm ở `work_dir/ccp_dbg`, không đụng vào Drive.

In [ ]:
%cd /content/thesis/SIP-HyperGCN
!python scripts/train_ccp.py $COMMON --out_dir work_dir/ccp_dbg --stages 1 2 3 \
    --epochs1 1 --epochs2 1 --epochs3 1 --max_train_batches 10 --max_val_batches 3 \
    --calib_batches 5 --log_every 5 --batch_size 32
!python scripts/eval_ccp.py $COMMON --ccp_ckpt work_dir/ccp_dbg/ccp_stage3.pt --out_dir work_dir/ccp_dbg/eval \
    --max_samples 256 --cf_max_literals 1 --cf_per_literal 8 --cf_steps 5 --robust --batch_size 32

/content/thesis/SIP-HyperGCN
[load_hypergcn] using model.hypergcn_base.Model
[load_hypergcn] loaded /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt (607 tensors)
[build_feeder] split=train args={'data_path': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz', 'split': 'train', 'window_size': 64, 'p_interval': [0.95], 'vel': False, 'bone': False, 'debug': False, 'random_rot': False, 'random_choose': False, 'random_shift': False, 'random_move': False}


## 1. Giai đoạn 1: concept (ngưỡng τ, concept ẩn)
Hiệu chỉnh tín hiệu rồi học concept theo CSV (MIL). HyperGCN đóng băng.

In [ ]:
%%bash
python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 1

[load_hypergcn] using model.hypergcn_base.Model
[load_hypergcn] loaded /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt (607 tensors)
[build_feeder] split=train args={'data_path': '/content/data/NTU60_CS.npz', 'split': 'train', 'window_size': 64, 'p_interval': [0.95], 'vel': False, 'bone': False, 'debug': False, 'random_rot': False, 'random_choose': False, 'random_shift': False, 'random_move': False}


Traceback (most recent call last):
  File "/content/thesis/SIP-HyperGCN/scripts/train_ccp.py", line 300, in <module>
    main()
    ~~~~^^
  File "/content/thesis/SIP-HyperGCN/scripts/train_ccp.py", line 268, in main
    fit_loader, val_loader = make_loaders(arg)
                             ~~~~~~~~~~~~^^^^^
  File "/content/thesis/SIP-HyperGCN/scripts/train_ccp.py", line 85, in make_loaders
    train_set = build_feeder(arg.hypergcn_root, arg.data_path, "train", config=arg.config)
  File "/content/thesis/SIP-HyperGCN/sip/backbone.py", line 74, in build_feeder
    return Feeder(**args)
  File "/content/thesis/GCCR-Skateformer-TreeRule/hypergcn/feeders/feeder_ntu.py", line 50, in __init__
    self.load_data()
    ~~~~~~~~~~~~~~^^
  File "/content/thesis/GCCR-Skateformer-TreeRule/hypergcn/feeders/feeder_ntu.py", line 56, in load_data
    npz_data = np.load(self.data_path)
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 476, in load
    ret = NpzFile(fid, o

CalledProcessError: Command 'b'python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 1\n'' returned non-zero exit status 1.

## 2. Giai đoạn 2: chương trình logic
Theo dõi `VAL logic=`: accuracy khi chỉ dùng tầng logic.

In [ ]:
%%bash
python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 2 --init_ckpt $V4/ccp_stage1.pt

## 3. Giai đoạn 3: kết hợp + nhất quán phản thực tế
Fine-tune HyperGCN (lr nhỏ) với L_cf. Thêm `--freeze_backbone` nếu muốn giữ nguyên HyperGCN (khi đó L_cf tắt).

In [ ]:
%%bash
python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 3 --init_ckpt $V4/ccp_stage2.pt

## 4. Đánh giá trên TEST

In [ ]:
%%bash
python scripts/eval_ccp.py $COMMON --ccp_ckpt $V4/ccp_stage3.pt --out_dir $V4/eval --robust

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(os.environ["V4"], "eval", "report.md"), encoding="utf-8").read()))